In [1]:
import pandas as pd
import numpy as np

In [5]:
df=pd.read_csv('phishing_email.csv')

In [7]:
df.column

AttributeError: 'DataFrame' object has no attribute 'column'

In [9]:
df.head()

,text_combined,label
0,hpl nom may 25 2001 see attached file hplno 52...,0
1,nom actual vols 24 th forwarded sabrae zajac h...,0
2,enron actuals march 30 april 1 201 estimated a...,0
3,hpl nom may 30 2001 see attached file hplno 53...,0
4,hpl nom june 1 2001 see attached file hplno 60...,0


In [19]:
df.shape


(82486, 2)

In [21]:
df.columns

Index(['text_combined', 'label'], dtype='object')

In [25]:
df['label'].value_counts()

label
1    42891
0    39595
Name: count, dtype: int64

In [27]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)   # remove URLs
    text = re.sub(r'[^a-z\s]', '', text)          # remove punctuation/numbers
    text = re.sub(r'\s+', ' ', text).strip()      # remove extra whitespace
    return text

df['clean_text'] = df['text_combined'].apply(clean_text)

In [28]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)

In [30]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [31]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [32]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, classification_report

y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.9812098436174082

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.98      0.98      7919
           1       0.98      0.98      0.98      8579

    accuracy                           0.98     16498
   macro avg       0.98      0.98      0.98     16498
weighted avg       0.98      0.98      0.98     16498


Confusion Matrix:
 [[7754  165]
 [ 145 8434]]


In [37]:
import numpy as np

feature_names = np.array(vectorizer.get_feature_names_out())
coefs = model.coef_[0]

top_phishing_idx = np.argsort(coefs)[-20:][::-1]
top_legit_idx = np.argsort(coefs)[:20]

print("Top words indicating PHISHING:")
print(feature_names[top_phishing_idx])

print("\nTop words indicating LEGITIMATE:")
print(feature_names[top_legit_idx])

Top words indicating PHISHING:
['josemonkeyorg' 'aug' 'account' 'http' 'life' 'remove' 'love' 'click'
 'thu' 'money' 'investment' 'meds' 'watches' 'cnncom' 'company' 'bank'
 'statements' 'wed' 'viagra' 'replica']

Top words indicating LEGITIMATE:
['wrote' 'enron' 'thanks' 'pm' 'opensuse' 'ierant' 'vince' 'university'
 'louise' 'rssfeedsspamassassintaintorg' 'im' 'url' 'questions' 'python'
 'tony' 'perl' 'edu' 'fork' 'date' 'attached']
